# Inspect Aggregated Runoff Datasets

HRU-level inspection of the two runoff sources, aggregated to the fabric. Mirrors `inspect_consolidated_runoff.ipynb` at the HRU level.

Sources (see `catalog/variables.yml` → `runoff`):

- ERA5-Land total runoff (`ro`, m/month)
- GLDAS-2.1 NOAH total runoff (`runoff_total = Qs_acc + Qsb_acc`, kg m⁻² stored as the mean of 3-hourly accumulations; multiply by `8 × days_in_month` to get mm/month)

See `docs/references/calibration-target-recipes.md` §1 for the canonical-unit conversions and combination rule.

## Per-target conventions in this notebook

- ERA5-Land `ro` × 1000 → mm/month before normalised comparison.
- GLDAS NOAH `runoff_total` × 8 × `days_in_month` → mm/month (recipes §1, lesson 3). The aggregated NC still carries `kg m⁻²` units; the conversion happens in this notebook before plotting.
- Reference source for the colour scale and footprint-clip: ERA5-Land.
- ERA5-Land timestamps are end-of-month; GLDAS at start-of-month. We use `select_month` rather than `time.sel(method="nearest")` for cross-source alignment (recipes lesson 2).
- Units are read from `catalog/sources.yml` via `unit_from_catalog`, not from the on-disk NC `attrs` (recipes lesson 9).

In [ ]:
import calendar
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from _helpers import (
    area_weighted_mean,
    discover_aggregated,
    load_fabric,
    load_project_paths,
    lookup_hrus_by_points,
    nan_hru_count,
    open_year,
    open_year_range,
    plot_hru_choropleth,
    plot_nan_hrus,
    save_figure,
    select_month,
    unit_from_catalog,
)

# Edit me to point at a real project directory:
PROJECT_DIR = Path(
    "/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets"
)

# Set True (and re-run) to populate docs/figures/inspect_aggregated/<run>_*.png
# import _helpers
# _helpers.SAVE_FIGURES = True

project_dir, datastore_dir, fabric_cfg = load_project_paths(PROJECT_DIR)
fabric = load_fabric(fabric_cfg)

TARGET = "runoff"
TARGET_TIME = "2000-01-15"
TARGET_YEAR = 2000
TARGET_MONTH = 1
TIME_SERIES_YEARS = range(2000, 2011)
REPRESENTATIVE_POINTS = {
    "Olympic Peninsula (PNW mountains)": (-123.5, 47.8),
    "Iowa cropland (Midwest)": (-93.6, 42.0),
    "Phoenix metro (arid SW)": (-112.1, 33.4),
    "Southern Appalachians (Eastern forest)": (-83.5, 35.5),
}

SOURCES = {
    "era5_land": {"label": "ERA5-Land (ro)", "var": "ro"},
    "gldas_noah_v21_monthly": {"label": "GLDAS-2.1 NOAH (runoff_total)", "var": "runoff_total"},
}

print(f"Project: {project_dir}")
print(f"Datastore: {datastore_dir}")
print(f"Fabric: {fabric_cfg['path']} ({len(fabric)} HRUs)")

## Load aggregated datasets

Each source is opened from `<project>/data/aggregated/<source_key>/<source_key>_<TARGET_YEAR>_agg.nc`. Sources whose aggregation has not yet been produced are skipped with a clear message; downstream cells iterate over the loaded set so missing sources drop out naturally.

In [ ]:
opened = {}
for source_key, info in SOURCES.items():
    paths = discover_aggregated(project_dir, source_key)
    if paths is None:
        print(f"SKIP {info['label']}: no aggregated NCs at "
              f"{project_dir}/data/aggregated/{source_key}/")
        continue
    ds = open_year(project_dir, source_key, TARGET_YEAR)
    info["units"] = unit_from_catalog(source_key, info["var"])
    opened[source_key] = (ds, info)
    values = ds[info["var"]].isel(time=0).to_pandas()
    print(
        f"{info['label']}: vars={list(ds.data_vars)} | "
        f"time={ds.time.values[0]}..{ds.time.values[-1]} | "
        f"HRUs={ds.sizes.get('nhm_id', ds.sizes.get('hru_id', 'unknown'))} | "
        f"NaN HRUs (t=0): {nan_hru_count(values)} | "
        f"catalog units: {info['units']}"
    )

In [ ]:
for source_key, (ds, info) in opened.items():
    print(f"{'=' * 60}\n{info['label']}\n{'=' * 60}")
    display(ds)

In [ ]:
if not opened:
    print("No sources available; skipping native-unit map.")
else:
    fig, axes = plt.subplots(1, len(opened), figsize=(8 * len(opened), 6), squeeze=False)
    for idx, (source_key, (ds, info)) in enumerate(opened.items()):
        da = select_month(ds[info["var"]], TARGET_YEAR, TARGET_MONTH)
        values = da.to_pandas()
        plot_hru_choropleth(
            axes[0, idx],
            fabric,
            values,
            cmap="YlGnBu",
            title=f"{info['label']}\n{TARGET_TIME} | {info['units']}",
            units=info["units"],
        )
    fig.suptitle(f"Runoff — native units, {TARGET_TIME}", fontsize=13, y=1.02)
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_native_units_map")
    plt.show()

## Normalized comparison map (mm/month)

Convert both sources to **mm/month** and plot side by side on a shared colour scale derived from ERA5-Land (the reference source).

- ERA5-Land `ro` (m/month) × 1000 → mm/month.
- GLDAS NOAH `runoff_total` (kg m⁻², mean of 3-hourly accumulations) × `8 × days_in_month` → mm/month.

Both panels are on the HRU fabric, so the geographic footprint is identical; the colour scale is anchored on ERA5-Land's distribution to avoid GLDAS's tropical maxima dominating it.

In [ ]:
def _to_mm_per_month(da: xr.DataArray, source_key: str) -> xr.DataArray:
    if source_key == "era5_land":
        return da * 1000.0
    if source_key == "gldas_noah_v21_monthly":
        ts = pd.Timestamp(da.time.values)
        days = calendar.monthrange(ts.year, ts.month)[1]
        return da * 8.0 * days
    raise ValueError(f"No mm/month conversion for {source_key}")


if opened:
    converted = {}
    for source_key, (ds, info) in opened.items():
        da = select_month(ds[info["var"]], TARGET_YEAR, TARGET_MONTH)
        converted[source_key] = _to_mm_per_month(da, source_key).to_pandas()

    ref_key = "era5_land"
    if ref_key in converted:
        ref_vals = converted[ref_key].dropna().values
        vmin, vmax = float(np.percentile(ref_vals, 2)), float(np.percentile(ref_vals, 98))
    else:
        all_vals = np.concatenate([s.dropna().values for s in converted.values()])
        vmin, vmax = float(np.percentile(all_vals, 2)), float(np.percentile(all_vals, 98))

    fig, axes = plt.subplots(1, len(converted), figsize=(8 * len(converted), 6), squeeze=False)
    for idx, (source_key, values) in enumerate(converted.items()):
        plot_hru_choropleth(
            axes[0, idx],
            fabric,
            values,
            vmin=vmin,
            vmax=vmax,
            cmap="YlGnBu",
            title=f"{SOURCES[source_key]['label']}\n{TARGET_TIME} | mm/month",
            units="mm/month",
        )
    fig.suptitle(
        f"Runoff — mm/month, colour scale from ERA5-Land — {TARGET_TIME}",
        fontsize=13, y=1.02,
    )
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_normalized_comparison")
    plt.show()

In [ ]:
if opened:
    fig, ax = plt.subplots(figsize=(10, 5))
    for source_key, values in converted.items():
        ax.hist(
            values.dropna(),
            bins=60,
            histtype="step",
            label=SOURCES[source_key]["label"],
            density=True,
            linewidth=2,
        )
    ax.set_xlabel("Runoff (mm/month)")
    ax.set_ylabel("Density")
    ax.set_title(f"Cross-source HRU distribution — {TARGET_TIME}")
    ax.legend()
    save_figure(fig, f"{TARGET}_histogram")
    plt.show()

In [ ]:
if opened:
    rep_hrus = lookup_hrus_by_points(fabric, REPRESENTATIVE_POINTS)
    print("Representative HRUs:", rep_hrus)

    series_data = {}
    for source_key, info in SOURCES.items():
        if source_key not in opened:
            continue
        ds_range = open_year_range(project_dir, source_key, TIME_SERIES_YEARS)
        try:
            id_dim = "nhm_id" if "nhm_id" in ds_range.dims else "hru_id"
            sel = ds_range[info["var"]].sel({id_dim: list(rep_hrus.values())}).load()
        finally:
            ds_range.close()
        series_data[source_key] = sel

    def _convert_series(da, source_key):
        if source_key == "era5_land":
            return da * 1000.0
        if source_key == "gldas_noah_v21_monthly":
            days = pd.DatetimeIndex(da.time.values).days_in_month
            return da * 8.0 * xr.DataArray(days.values, coords={"time": da.time}, dims=["time"])
        return da

    fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, sharey=True)
    id_dim = "nhm_id" if "nhm_id" in next(iter(series_data.values())).dims else "hru_id"
    for ax, (label, hru_id) in zip(axes.flat, rep_hrus.items()):
        for source_key, da in series_data.items():
            da_hru = _convert_series(da.sel({id_dim: hru_id}), source_key)
            ax.plot(da_hru.time, da_hru.values, label=SOURCES[source_key]["label"])
        ax.set_title(f"{label} (HRU {hru_id})")
        ax.set_ylabel("mm/month")
        ax.legend(fontsize=8)
    fig.suptitle(f"Runoff at representative HRUs — {min(TIME_SERIES_YEARS)}–{max(TIME_SERIES_YEARS)}")
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_time_series")
    plt.show()

In [ ]:
if opened:
    fig, axes = plt.subplots(1, len(opened), figsize=(8 * len(opened), 6), squeeze=False)
    for idx, (source_key, (ds, info)) in enumerate(opened.items()):
        da = select_month(ds[info["var"]], TARGET_YEAR, TARGET_MONTH)
        values = da.to_pandas()
        n_nan = nan_hru_count(values)
        print(f"{info['label']}: {n_nan} NaN HRUs ({100 * n_nan / len(fabric):.2f}%)")
        plot_nan_hrus(
            axes[0, idx],
            fabric,
            values,
            title=f"{info['label']}\nNaN HRUs (red) — {n_nan} of {len(fabric)}",
        )
    fig.suptitle(
        "Coverage gaps — these will be nearest-neighbor-filled in normalize/ before target combination",
        fontsize=12,
        y=1.02,
    )
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_coverage")
    plt.show()

In [ ]:
def _gridded_mean_runoff(source_key, info):
    """Compute the gridded CONUS-footprint mean for the validation cell."""
    if source_key == "era5_land":
        path = datastore_dir / "era5_land" / "monthly" / f"era5_land_monthly_{TARGET_YEAR}.nc"
    elif source_key == "gldas_noah_v21_monthly":
        path = datastore_dir / "gldas_noah_v21_monthly" / "gldas_noah_v21_monthly.nc"
    else:
        return None, f"unknown gridded path for {source_key}"
    if not path.exists():
        return None, f"missing consolidated NC at {path}"
    with xr.open_dataset(path) as ds:
        da = select_month(ds[info["var"]], TARGET_YEAR, TARGET_MONTH).load()
    return _to_mm_per_month(da, source_key).mean(skipna=True).item(), None


print(f"{'Source':<35} {'Aggregated':>12} {'Gridded':>12} {'Δ':>12} {'% diff':>8}")
print("-" * 85)
for source_key, (ds, info) in opened.items():
    da_agg = select_month(ds[info["var"]], TARGET_YEAR, TARGET_MONTH)
    converted_agg = _to_mm_per_month(da_agg, source_key).to_pandas()
    agg_mean = area_weighted_mean(converted_agg, fabric)
    gridded_mean, reason = _gridded_mean_runoff(source_key, info)
    if gridded_mean is None:
        print(f"{info['label']:<35} {agg_mean:>12.3f}  {'SKIP':>12} ({reason})")
        continue
    diff = agg_mean - gridded_mean
    pct = 100 * diff / gridded_mean if gridded_mean else float("nan")
    print(f"{info['label']:<35} {agg_mean:>12.3f} {gridded_mean:>12.3f} {diff:>12.3f} {pct:>7.2f}%")

## Why HRU-level patterns differ across sources

After area-weighted aggregation to HRUs, the HRU-level magnitudes track the gridded means within a few percent (validation cell above). Differences that remain are physical:

- **Different LSM physics.** ERA5-Land's H-TESSEL and GLDAS-2.1's NOAH-2.7 partition precipitation between infiltration, surface runoff, and sub-surface drainage with different parameterisations. Two LSMs forced with the same precipitation will still produce different runoff.
- **Different forcing.** ERA5-Land uses ERA5 precipitation; GLDAS-2.1 blends NOAA CPC gauge-corrected precipitation with model fields. Disagreement is largest in winter and over mountains.
- **Aggregation behaviour at coarse-grid sources.** GLDAS at ~25 km will yield more HRUs whose polygon does not have full source-grid coverage than ERA5-Land at ~9 km, so the NaN-HRU count is generally higher for GLDAS — visible in the coverage diagnostic above.

**Calibration target implication.** The runoff target uses both sources as a per-HRU per-month `min/max` range. Both must be in mm/month before the cfs conversion (`run.py`) — the GLDAS unit fix is critical, since omitting it makes GLDAS values 224–248× too small and degenerates the multi-source range.

In [ ]:
for source_key, (ds, _) in opened.items():
    ds.close()
opened.clear()